# A.12.2 Parameter data

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

There are two forms of the statement that specifies parameter data or variable initial values.
The first form is analogous to the set data statement:

```ampl
param - data - statement:
     param param - name param - default opt := param - spec param - spec . . . ;
param - spec:
     param - template opt value - list
     param - template opt value - table value - table . . .
```

with the addition of an optional `param` - default that will be described below. The `param` - name is
usually the name of a parameter declared in the model, but may also be the name of a variable or
constraint; the keyword `var` may be used instead of `param` to make the distinction clear.

The `param` statement's templates have the same content as in the set data statement, but are
given in brackets (like subscripts) rather than parentheses:

```ampl
param - template:
     [ templ - item, templ - item, . . . ]
templ - item:
      ob j ect
      *
      :
```

The value - list is like the previously defined member - list, except that it also specifies a parameter
or variable value:

```ampl
value - list:
       value - item value - item . . .
value - item:
       ob j ect ob j ect . . . entry
```

The objects are substituted for \*'s in the template to define a set member, and the parameter or
variable indexed by this set member is assigned the value associated with the entry (see below).
The value - `table` is like the previously defined member - `table`, except that its entrys are values rather
than + or -:

```ampl
value - table:
       (tr) opt            t - header   t - header   t - header   ...   :=
       row - label                entry        entry        entry
       row - label                entry        entry        entry
       ...
t - header:
       : ob j ect      ob j ect    ...
row - label:
       ob j ect   ob j ect   ...
entry:
      number
      string
      default - symbol
```

As in set statements, the notation (tr) means transpose; it implies a 2D `table`, and a : after
it is optional. It remains in effect until a new template appears.

A `table` may be given in several chunks of columns.

Each entry's row - label and `t` - header entries are substituted for \*'s and :'s in the template to
define a set member, and the parameter or variable indexed by this set member is assigned the
value specified by the entry. The entry may be a number for variables and for parameters that take
numerical values, or a string for variables and parameters declared with the attribute symbolic.
An entry that is the default symbol (see below) is ignored.

The second form of parameter data statement provides for the definition of multiple parameters,
and also optionally the set over which they are indexed:

```ampl
param - data - alternate:
     param param - default opt :
            param - name param - name . . . := value - item value - item . . . ;
       param param - default opt : set - name:
           param - name param - name . . . := value - item value - item . . . ;
```

The named parameters must all have the same dimension. If the optional set - name is specified, its
membership is also defined by this statement. Each value - item consists of an optional template
followed by a list of objects and a list of values:

```ampl
value - item:
       template opt ob j ect . . . entry entry . . .
```

An initial template of all \*'s (as many as the common dimension of the named parameters) is
assumed, and a template remains in effect until a new one appears. The objects must be equal in
number to the number of \*'s in the current template; when substituted for the \*'s in the current
template, they define a set member. If a set is being defined, this member is added to it. The
parameters indexed by this member are assigned the values associated with the subsequent entrys,
which obey the same rules as the `table` entrys previously described. Values are assigned in the
order in which the parameters' names appeared at the beginning of the statement; the number of
entrys must equal the number of named parameters.

A `param` data statement's optional default phrase has the form

```ampl
param - default:
     default number
```

If this phrase is present, any parameter named but not explicitly assigned a value in the statement is
given the value of number.

A data item may be specified as "." rather than an explicit value. This is normally taken as a
missing value, and a reference to it in the model would cause an error message. If there is a
default value, however, each missing value is determined from that default. A default value
may be specified either through a default phrase in a parameter's declaration in the model (A.7), or
from an optional phrase

```ampl
default r
```

that follows the parameter's name in the data statement. In the latter case, r must be a numeric
constant.

Default-value symbols may appear in both 1D and 2D tables. The default - symbol is initially a
dot (.). A stack of default-value symbols is maintained, with the current symbol at the top. The
defaultsym statement (which is recognized only in a data section) pushes a new symbol onto
the stack, and nodefaultsym pushes a "no symbol" indicator onto the stack. The statement

```ampl
defaultsym;
```

(without a symbol) pops the stack.

Parameters having three or more indices may be given values by a sequence of 1D and 2D
tables, one or more for each slice of nondefault values.

In summary, a `param` statement defining one indexed parameter starts with the keyword
`param` and the name of the parameter, followed by an optional default value and an optional :=.
Then comes a sequence of 1D and 2D `param` tables, which are similar to 1D and 2D set tables,
except that templates involve square brackets rather than parentheses, 2D tables contain numbers
(or, for a symbolic parameter, literals) rather than +'s and -'s, and 1D tables corresponding to a
template of k \*'s contain k + 1 rather than k columns, the last being a column of numbers or
default symbols (.'s). A special form, the keyword `param`, an optional default value, and a single
(untransposed) 2D `table`, defines several parameters indexed by a common set, and another special
form, `param` followed by the parameter name, an optional :=, and a numeric value, defines a
scalar parameter.

Variable and constraint names may appear in data sections anywhere that a parameter name
may appear, to specify initial values for variables and for the dual variables associated with constraints.
The rules for default values are the same as for parameters. The keyword `var` is a synonym
for `param` in data statements.